# paragami: folding and flattening parameters

paragami describes the *shape and constraints* of a parameter with a **pattern**. A pattern moves a value between three representations:

| representation | what it is | example for a covariance matrix |
|---|---|---|
| **folded** | the value as you'd write it in code (an array, or a dict of arrays) | a $K \times K$ positive definite matrix |
| **flat** | the folded value's entries as one vector, constraints and all | its $K^2$ entries |
| **free** | an unconstrained vector of the right dimension | the $K(K+1)/2$ entries of the log-Cholesky factor |

The free representation is what optimizers and samplers want. paragami uses Stan's unconstraining transforms, so free vectors and log Jacobians match Stan's unconstrained space exactly. Everything is pure JAX, so it works under `grad`, `jit` and `vmap`.

Run `./setup_venv.sh` in this folder once and register the kernel it prints; this notebook uses that kernel ("Python (modelgami)").

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import paragami as pg

jax.config.update("jax_enable_x64", True)

## A single pattern

A positive scalar, and a vector bounded to $(0, 1)$. `free=True` gives the unconstrained vector; `free=False` gives the flat vector.

In [2]:
sigma_pattern = pg.NumericScalarPattern(lb=0.0)
x = sigma_pattern.flatten(2.0, free=True)
print("free value:", x, "  (log 2 =", np.log(2.0), ")")
print("folded back:", sigma_pattern.fold(x, free=True))

p_pattern = pg.NumericVectorPattern(3, lb=0.0, ub=1.0)
p = jnp.array([0.1, 0.5, 0.9])
print("free value of p:", p_pattern.flatten(p, free=True), " (the logit)")

free value: [0.69314718]   (log 2 = 0.6931471805599453 )
folded back: 2.0


free value of p: [-2.19722458  0.          2.19722458]  (the logit)


## Structured constraints

Some patterns have a free dimension smaller than their flat dimension. A simplex in $K$ dimensions has $K - 1$ free parameters, and a $K \times K$ covariance matrix has $K(K+1)/2$.

In [3]:
patterns = {
    "simplex(4)": pg.SimplexArrayPattern(4),
    "ordered(3)": pg.OrderedVectorPattern(3),
    "positive_ordered(3)": pg.OrderedVectorPattern(3, positive=True),
    "cov_matrix(3)": pg.PSDSymmetricMatrixPattern(3),
    "cholesky_factor_cov(3)": pg.CholeskyCovPattern(3),
    "cholesky_factor_corr(3)": pg.CholeskyCorrPattern(3),
}
for name, pat in patterns.items():
    print(f"{name:25s} flat length {pat.flat_length(free=False):2d}   free length {pat.flat_length(free=True):2d}")

simplex(4)                flat length  4   free length  3
ordered(3)                flat length  3   free length  3
positive_ordered(3)       flat length  3   free length  3
cov_matrix(3)             flat length  9   free length  6
cholesky_factor_cov(3)    flat length  9   free length  6
cholesky_factor_corr(3)   flat length  9   free length  3


In [4]:
cov_pattern = pg.PSDSymmetricMatrixPattern(3)
u = jax.random.normal(jax.random.PRNGKey(0), (cov_pattern.flat_length(free=True),))
cov = cov_pattern.fold(u, free=True)    # any free vector folds to a valid covariance matrix
print(cov)
print("eigenvalues:", jnp.linalg.eigvalsh(cov))
print("round trip error:", jnp.max(jnp.abs(cov_pattern.flatten(cov, free=True) - u)))

[[ 0.66253342 -0.63876917  0.15289781]
 [-0.63876917 38.41072605  0.34974222]
 [ 0.15289781  0.34974222  0.51692897]]
eigenvalues: [ 0.40961455  0.75586342 38.42471047]


round trip error: 2.7755575615628914e-17


## Dictionaries of parameters

`PatternDict` combines patterns, and its folded values are dicts. This is how a model's parameters are described in modelgami.

In [5]:
pattern = pg.PatternDict(
    mu=pg.NumericVectorPattern(3),
    sigma=pg.NumericScalarPattern(lb=0.0),
    w=pg.SimplexArrayPattern(4),
    cov=pg.PSDSymmetricMatrixPattern(2),
)
params = pattern.random(jax.random.PRNGKey(1))
params

{'mu': Array([-1.18428442, -0.11617041,  0.17269028], dtype=float64),
 'sigma': Array(2.6046731, dtype=float64),
 'w': Array([0.15956632, 0.51823804, 0.12339456, 0.19880108], dtype=float64),
 'cov': Array([[ 0.21674056, -0.02357817],
        [-0.02357817,  0.06943873]], dtype=float64)}

In [6]:
x_free = pattern.flatten(params, free=True)
x_flat = pattern.flatten(params, free=False)
print("free length:", x_free.shape[0], "  flat length:", x_flat.shape[0])
print(pattern.flat_names(free=True))

free length: 10   flat length: 12
['mu[0]', 'mu[1]', 'mu[2]', 'sigma', 'w[0]', 'w[1]', 'w[2]', 'cov[0]', 'cov[1]', 'cov[2]']


## Log Jacobians

A density over the constrained parameters becomes a density over the free vector by adding the log absolute determinant of the free-to-constrained Jacobian. `fold_with_log_det` returns both at once. Here we check it against autodiff for the simplex, where the free and flat dimensions differ, so we use the flat value without its last (redundant) entry.

In [7]:
simplex = pg.SimplexArrayPattern(4)
u = jnp.array([0.3, -1.0, 0.5])
w, log_det = simplex.fold_with_log_det(u)

J = jax.jacfwd(lambda v: simplex.fold(v, free=True)[:-1])(u)
print("paragami log det:", log_det)
print("autodiff log det:", jnp.linalg.slogdet(J)[1])

paragami log det: -5.6088912964925095
autodiff log det: -5.6088912964925095


## Validation

`validate_folded` checks keys, shapes and constraint values and returns `(ok, message)`. `fold` and `flatten` validate their inputs; inside `jit`, value checks are skipped automatically and shape checks remain.

In [8]:
bad = dict(params, sigma=-1.0)
print(pattern.validate_folded(bad))
print(pattern.validate_folded(dict(params, w=jnp.ones(3) / 3)))
try:
    pattern.flatten(bad, free=True)
except Exception as e:
    print(type(e).__name__, ":", e)

(False, "PatternDict(patterns={'mu': ('NumericVectorPattern', {'shape': (3,), 'lb': -inf, 'ub': inf, 'order': 'C'}), 'sigma': ('NumericScalarPattern', {'shape': (), 'lb': 0.0, 'ub': inf, 'order': 'C'}), 'w': ('SimplexArrayPattern', {'simplex_size': 4, 'array_shape': ()}), 'cov': ('PSDSymmetricMatrixPattern', {'size': 2})}): ['sigma'] values must be > lb = 0.0 (min -1.0)")
(False, "PatternDict(patterns={'mu': ('NumericVectorPattern', {'shape': (3,), 'lb': -inf, 'ub': inf, 'order': 'C'}), 'sigma': ('NumericScalarPattern', {'shape': (), 'lb': 0.0, 'ub': inf, 'order': 'C'}), 'w': ('SimplexArrayPattern', {'simplex_size': 4, 'array_shape': ()}), 'cov': ('PSDSymmetricMatrixPattern', {'size': 2})}): ['w'] expected shape (4,), got (3,)")
ValidationError : PatternDict(patterns={'mu': ('NumericVectorPattern', {'shape': (3,), 'lb': -inf, 'ub': inf, 'order': 'C'}), 'sigma': ('NumericScalarPattern', {'shape': (), 'lb': 0.0, 'ub': inf, 'order': 'C'}), 'w': ('SimplexArrayPattern', {'simplex_size': 4, 

## Arrays of patterns

`PatternArray` repeats a pattern over an array of shape `array_shape`, for example one simplex per group.

In [9]:
group_weights = pg.PatternArray((5,), pg.SimplexArrayPattern(3))
print("free length:", group_weights.flat_length(free=True))
W = group_weights.random(jax.random.PRNGKey(2))
print(W.shape, W.sum(axis=-1))

free length: 10


(5, 3) [1. 1. 1. 1. 1.]


## Using patterns with JAX transformations

A function of folded parameters becomes a function of the free vector with `FoldFunctionInput`. You can then differentiate, compile and vectorize it.

In [10]:
def objective(params):
    return jnp.sum(params["mu"] ** 2) + params["sigma"] + jnp.sum(params["w"] * jnp.arange(4.0)) + jnp.trace(params["cov"])

f = pg.FoldFunctionInput(objective, pattern, free=True)
print("f(x):", f(x_free))
print("grad:", jax.jit(jax.grad(f))(x_free))

xs = jax.random.normal(jax.random.PRNGKey(3), (10, x_free.shape[0]))
print("vmap over 10 free vectors:", jax.vmap(f)(xs).shape)

f(x): 5.698129887361914


grad: [-2.36856884 -0.23234082  0.34538056  2.6046731  -0.02116473 -0.22949165
 -0.37614301  0.43348113 -0.10129078  0.13374755]


vmap over 10 free vectors: (10,)


A pattern acts on a single value, so batches of draws are folded with `jax.vmap`. The result is a dict of arrays with a leading batch axis. Notebook 3 uses this to turn MCMC output into structured draws.

In [11]:
folded_batch = jax.vmap(lambda v: pattern.fold(v, free=True))(xs)
{k: v.shape for k, v in folded_batch.items()}

{'cov': (10, 2, 2), 'mu': (10, 3), 'sigma': (10,), 'w': (10, 4)}